In [0]:
import subprocess

from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)


# /tmp does not support the Unix-socket link operation in this environment.
# /dev/shm is a local tmpfs and supports SSH multiplexing sockets.
CONTROL_SOCKET = "/dev/shm/databricks-ssh"

SSH_HOST = "databricks@pub.worldb.dedyn.io"
SSH_PORT = 8889
SSH_KEY = (
    "/Workspace/Users/rogermm@gmail.com/"
    ".ssh/id_ed25519_databricks_free"
)

KNOWN_HOSTS_FILE = "/tmp/known_hosts"

tools_host = "172.18.0.8"
# tools_host = "tools"

host_ip = "localhost"
# host_ip = "192.168.210.33"


ssh_base = (
    f"ssh "
    f"-S {CONTROL_SOCKET} "
    f"-p {SSH_PORT} "
    f"-i {SSH_KEY} "
    f"-o BatchMode=yes "
    f"-o StrictHostKeyChecking=accept-new "
    f"-o UserKnownHostsFile={KNOWN_HOSTS_FILE} "
)


def start_ssh_tunnel():
    command = (
        f"ssh "
        f"-M "
        f"-S {CONTROL_SOCKET} "
        f"-o ControlPersist=yes "
        f"-o ServerAliveInterval=30 "
        f"-o ServerAliveCountMax=3 "
        f"-o ExitOnForwardFailure=yes "
        f"-o StrictHostKeyChecking=accept-new "
        f"-o UserKnownHostsFile={KNOWN_HOSTS_FILE} "
        f"-o BatchMode=yes "
        f"-p {SSH_PORT} "
        f"-i {SSH_KEY} "
        f"-L 127.0.0.1:9092:kafka-4:9092 "
        f"-L 127.0.0.1:8080:traefik:80 "
        f"-L {host_ip}:8888:{tools_host}:8888 "
        f"-fNT "
        f"{SSH_HOST}"
    )

    result = subprocess.run(
        command,
        shell=True,
        text=True,
        capture_output=True,
        check=False,
    )

    return command, result


def check_ssh_tunnel():
    # -O must appear before the SSH destination.
    command = f"{ssh_base}-O check {SSH_HOST}"

    result = subprocess.run(
        command,
        shell=True,
        text=True,
        capture_output=True,
        check=False,
    )

    return command, result


def stop_ssh_tunnel():
    # -O must appear before the SSH destination.
    command = f"{ssh_base}-O exit {SSH_HOST}"

    result = subprocess.run(
        command,
        shell=True,
        text=True,
        capture_output=True,
        check=False,
    )

    return command, result


def remove_stale_control_socket():
    command = (
        f"rm -f "
        f"{CONTROL_SOCKET} "
        f"{CONTROL_SOCKET}.*"
    )

    return subprocess.run(
        command,
        shell=True,
        text=True,
        capture_output=True,
        check=False,
    )


tunnel_result_schema = StructType(
    [
        StructField("success", BooleanType(), nullable=False),
        StructField("action", StringType(), nullable=False),
        StructField("worker", StringType(), nullable=False),
        StructField("exit_code", IntegerType(), nullable=False),
        StructField("command", StringType(), nullable=False),
        StructField("stdout", StringType(), nullable=False),
        StructField("stderr", StringType(), nullable=False),
    ]
)


@F.udf(returnType=tunnel_result_schema)
def manage_ssh_tunnel(action):
    import socket

    normalized_action = (action or "").strip().lower()
    worker = socket.gethostname()

    try:
        if normalized_action == "start":
            check_command, check_result = check_ssh_tunnel()

            # Make the start operation idempotent.
            if check_result.returncode == 0:
                return (
                    True,
                    normalized_action,
                    worker,
                    check_result.returncode,
                    check_command,
                    check_result.stdout.strip(),
                    (
                        check_result.stderr.strip()
                        or "SSH tunnel is already running."
                    ),
                )

            # Remove an abandoned socket left by a previous SSH process.
            remove_stale_control_socket()

            command, result = start_ssh_tunnel()

        elif normalized_action == "check":
            command, result = check_ssh_tunnel()

        elif normalized_action == "stop":
            command, result = stop_ssh_tunnel()

        else:
            return (
                False,
                normalized_action,
                worker,
                -1,
                "",
                "",
                "Invalid action. Use start, check, or stop.",
            )

        return (
            result.returncode == 0,
            normalized_action,
            worker,
            result.returncode,
            command,
            result.stdout.strip(),
            result.stderr.strip(),
        )

    except Exception as error:
        return (
            False,
            normalized_action,
            worker,
            -1,
            "",
            "",
            f"{type(error).__name__}: {error}",
        )

In [0]:
worker_count = 1

start_result = (
    spark.range(worker_count)
    .repartition(worker_count)
    .withColumn(
        "result",
        manage_ssh_tunnel(F.lit("start")),
    )
    .select("id", "result.*")
)

start_result.show(truncate=False)


+---+-------+------+------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+---------------------------------------------------------------------------------------------+
|id |success|action|worker|exit_code|command                                                                                                                                                                                                                                                                                                                                                                                              

In [0]:
!ssh -M -S /tmp/databricks-ssh -p 8889 -o BatchMode=yes   -i /Workspace/Users/rogermm@gmail.com/.ssh/id_ed25519_databricks_free tunnel@pub.worldb.dedyn.io whoami


In [0]:
%%bash
ssh -M -S /tmp/databricks-ssh -p 8889 -i /Workspace/Users/rogermm@gmail.com/.ssh/id_ed25519_databricks_free \
    -o BatchMode=yes \
    -o ControlPersist=yes \
    -o ServerAliveInterval=30 \
    -o ServerAliveCountMax=3 \
    -o ExitOnForwardFailure=yes \
    -o StrictHostKeyChecking=accept-new \
    -o UserKnownHostsFile=/tmp/known_hosts \
    -fNT \
    tunnel@pub.worldb.dedyn.io

tunnel@pub.worldb.dedyn.io: Permission denied (publickey).


---------------------------------------------------------------------------
CalledProcessError                        Traceback (most recent call last)
File <command-4942731051954540>, line 1
----> 1 get_ipython().run_cell_magic('bash', '', 'ssh -M -S /tmp/databricks-ssh -p 8889 -i /Workspace/Users/rogermm@gmail.com/.ssh/id_ed25519_databricks_free \\\n    -o BatchMode=yes \\\n    -o ControlPersist=yes \\\n    -o ServerAliveInterval=30 \\\n    -o ServerAliveCountMax=3 \\\n    -o ExitOnForwardFailure=yes \\\n    -o StrictHostKeyChecking=accept-new \\\n    -o UserKnownHostsFile=/tmp/known_hosts \\\n    -fNT \\\n    tunnel@pub.worldb.dedyn.io\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # whe